# Ingest ParlGov data
This notebook downloads the ParlGov data bundle, harmonizes ISO3 codes,
and writes cleaned election, cabinet, and party tables for analysis.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.elections import (
    download_parlgov_codebook,
    download_parlgov_zip,
    load_parlgov_bundle,
    parlgov_metadata,
    write_parlgov_metadata,
    write_parlgov_outputs,
)
from src.paths import INTERMEDIATE_DIR, PAPER_LOGS_DIR, RAW_DIR
from src.qc import assert_unique_key
from src.viz_style import set_style

In [2]:
set_style()
raw_dir = RAW_DIR / "parlgov"

zip_path = download_parlgov_zip(raw_dir)
codebook_path = download_parlgov_codebook(raw_dir)

bundle = load_parlgov_bundle(zip_path)
paths = write_parlgov_outputs(bundle, INTERMEDIATE_DIR)

meta = parlgov_metadata(bundle, zip_path, codebook_path)
meta_path = PAPER_LOGS_DIR / "parlgov_metadata.json"
write_parlgov_metadata(meta, meta_path)

display(pd.DataFrame([meta["build"]["tables"]]).style.set_caption("ParlGov table sizes"))
display(bundle.elections.head(5).style.set_caption("ParlGov elections (sample)"))
display(bundle.parties.head(5).style.set_caption("ParlGov parties (sample)"))

,elections,election_results,cabinets,cabinet_parties,parties,countries
0,1027,9776,1619,4012,1707,37


,election_id,type_id,country_id,date,first_round_election_id,early,wikipedia,seats_total,electorate,votes_cast,votes_valid,data_source,description,comment,previous_parliament_election_id,previous_ep_election_id,previous_cabinet_id,old_countryID,old_parlID,country_name,iso3c,iso_numeric,election_date,year,election_type
0,1,13,11,1972-11-25,nan,0,"http://en.wikipedia.org/wiki/New_Zealand_general_election,_1972",87,1569937.000000,1410240.000000,1401152.000000,yb-nzl,nan,The sources are inconsistent in the number of seats. The official yearbook 1974 notifies 53 seats for Labour Party and 34 seats for National Party. The following yearbooks and Nohlen 2001 notifies the stated distribution.,120.000000,nan,569.000000,554.000000,19720.000000,New Zealand,NZL,554,1972-11-25 00:00:00,1972,Parliamentary election
1,2,13,64,1999-06-13,nan,1,"http://en.wikipedia.org/wiki/Belgian_federal_election,_1999",150,7343464.000000,6650015.000000,6214074.000000,ibz,nan,nan,332.000000,536.000000,95.000000,56.000000,19990.000000,Belgium,BEL,56,1999-06-13 00:00:00,1999,Parliamentary election
2,3,13,9,2005-09-12,nan,0,"http://en.wikipedia.org/wiki/Norwegian_parliamentary_election,_2005",169,3421741.000000,2649520.000000,2638263.000000,seby,nan,nan,328.000000,nan,300.000000,578.000000,20050.000000,Norway,NOR,578,2005-09-12 00:00:00,2005,Parliamentary election
3,4,1,23,2007-01-01,nan,0,"http://en.wikipedia.org/wiki/European_Parliament_election,_2007_(Romania)",35,nan,nan,nan,nan,nan,"Elections took only place later in 2007. Results reflect the initial composition of Romanian MEPs, based on the party composition of the Camera Deputaţilor.",165.000000,nan,28.000000,642.000000,1020070.000000,Romania,ROM,642,2007-01-01 00:00:00,2007,European Parliament
4,5,1,63,2009-06-07,nan,0,"http://en.wikipedia.org/wiki/European_Parliament_election,_2009_(Portugal)",22,9704559.000000,3568943.000000,3333195.000000,cne,nan,possible error in the primary source due to the valid votes,210.000000,140.000000,169.000000,620.000000,1020090.000000,Portugal,PRT,620,2009-06-07 00:00:00,2009,European Parliament


,party_id,country_id,family_id,name_short,name_english,name,name_ascii,name_nonlatin,wikipedia,data_source,description,comment,cmp,euprofiler,ees,morgan,castles_mair,huber_inglehart,ray,benoit_laver,chess,old_countryID,old_partyID,country_name,iso3c,iso_numeric,family_name,left_right,state_market,liberty_authority,eu_anti_pro
0,2,68,26,TOP09,Tradition Responsibility Prosperity 09,Tradice Odpovědnost Prosperita 09,Tradice Odpovednost Prosperita 09,nan,http://en.wikipedia.org/wiki/Tradition_Responsibility_Prosperity_09,nan,Party formed on 11 June 2009 as a split of the Christian Democratic Union (KDU-CSL) by MP Miroslav Kalousek.,nan,nan,nan,nan,nan,nan,nan,nan,nan,2109.000000,203.000000,14.000000,Czech Republic,CZE,203,Conservative,7.400000,6.400000,6.900000,7.900000
1,3,8,3,KNP,Catholic National Party,Katholieke Nationale Partij,Katholieke Nationale Partij,nan,http://en.wikipedia.org/wiki/Katholieke_Nationale_Partij,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,528.000000,46.000000,Netherlands,NLD,528,Christian democracy,6.200000,5.700000,7.100000,8.300000
2,4,5,26,PNP,People's New Party,Kokumin Shinto,Kokumin Shinto,国民新党,http://en.wikipedia.org/wiki/People%27s_New_Party,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,392.000000,38.000000,Japan,JPN,392,Conservative,7.400000,6.400000,6.900000,nan
3,5,23,26,PU|PC,Humanist Party | Conservative Party,Partidul Umanist | Partidul Conservator,Partidul Umanist | Partidul Conservator,nan,http://en.wikipedia.org/wiki/Conservative_Party_(Romania),nan,New party formed on 18 December 1991. Party was renamed into Conservative Party (PC) on 7 May 2005. PC merged with Liberal Reformist Party (PLR) into Alliance of Liberals and Democrats (ALDE) on 19 June 2015. PC members who did not want to merge into ALDE formed Social Liberal Humanist Party (PUSL) on 28 September 2015.,2015 change coded as one party to avoid high number of short lived parties in ParlGov,nan,nan,1642600.000000,nan,nan,nan,nan,3356.000000,2702.000000,642.000000,14.000000,Romania,ROM,642,Conservative,4.762200,4.753800,6.277900,8.202800
4,6,51,40,NO,New Horizons,Neoi Orizontes,Neoi Orizontes,Νέοι Ορίζοντες,http://en.wikipedia.org/wiki/New_Horizons_%28Cyprus%29,nan,New party formed in 1996.,nan,nan,nan,nan,nan,nan,nan,nan,5459.000000,nan,196.000000,31.000000,Cyprus,CYP,196,Right-wing,9.342100,6.578900,7.105300,nan


In [3]:
assert_unique_key(bundle.elections, ["election_id"])
assert_unique_key(bundle.parties, ["party_id"])
assert_unique_key(bundle.cabinets, ["cabinet_id"])

for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing expected output: {name}")

## Interpretation
The ParlGov tables now align with ISO3 country codes and are available as
parquet files for constructing the close-election running variable and
cabinet ideology diagnostics.